# Pill-Safe-AI (Colab Template)

이 노트북은 `colab_bundle.zip`을 업로드해서 FastAPI 백엔드를 Colab에서 실행하는 템플릿입니다.

## 순서
1) (로컬) `scripts/make_colab_bundle.ps1`로 `colab_bundle.zip` 생성
2) (Colab) `colab_bundle.zip` 업로드
3) 압축 해제 → 의존성 설치 → 서버 실행
4) 필요 시 ngrok로 외부 공개

## 0) 런타임 설정
- 상단 메뉴에서 **런타임 → 런타임 유형 변경 → GPU**(선택)
- 이 프로젝트는 GPU가 없어도 동작합니다(OCR 속도 차이).

In [ ]:
# 업로드한 colab_bundle.zip이 현재 디렉터리에 있다고 가정합니다
!ls -la

In [ ]:
# 압축 해제
!unzip -o colab_bundle.zip
!ls -la
!ls -la backend

In [ ]:
# Colab/Linux 친화 requirements 설치
!pip -q install -r backend/requirements-colab.txt

## (선택) MFDS/DUR API 키 주입 (Colab Secret 권장)

- Colab 좌측 **Secrets(열쇠 아이콘)** 에서 아래 이름으로 값을 등록한 뒤 실행하세요.
  - `MFDS_SERVICE_KEY`
  - `ODCLOUD_API_KEY`
- 키를 업로드 zip에 포함시키지 마세요(`backend/.env` 업로드 금지).

In [ ]:
import os

# Option A) Colab Secrets에서 가져오기 (권장)
try:
    from google.colab import userdata  # type: ignore

    mfds_key = userdata.get('MFDS_SERVICE_KEY')
    odcloud_key = userdata.get('ODCLOUD_API_KEY')
except Exception:
    mfds_key = None
    odcloud_key = None

# Option B) 직접 입력(임시)
# mfds_key = mfds_key or 'YOUR_MFDS_KEY'
# odcloud_key = odcloud_key or 'YOUR_ODCLOUD_KEY'

if mfds_key:
    os.environ['MFDS_SERVICE_KEY'] = mfds_key
if odcloud_key:
    os.environ['ODCLOUD_API_KEY'] = odcloud_key

# 필요하면 runtime 내에서만 backend/.env를 만들어 둘 수도 있음 (공유/커밋 금지)
# open('backend/.env', 'w', encoding='utf-8').write(
#     f"MFDS_SERVICE_KEY={os.environ.get('MFDS_SERVICE_KEY','')}\n"
#     f"ODCLOUD_API_KEY={os.environ.get('ODCLOUD_API_KEY','')}\n"
# )

print('MFDS_SERVICE_KEY set:', bool(os.environ.get('MFDS_SERVICE_KEY')))
print('ODCLOUD_API_KEY set:', bool(os.environ.get('ODCLOUD_API_KEY')))


In [ ]:
# (선택) GPU 런타임에서 torch 점검
import torch
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device', torch.cuda.get_device_name(0))

In [ ]:
# FastAPI 서버 실행 (백그라운드)
!python -m uvicorn backend.main:app --host 0.0.0.0 --port 8000 &

In [ ]:
# 헬스체크 / GPU 상태 확인
!curl -s http://127.0.0.1:8000/health
print()
!curl -s http://127.0.0.1:8000/gpu/status

## 1) (선택) ngrok로 외부 공개
Colab 내부 서버는 기본적으로 외부에서 접근이 어렵습니다. 모바일/외부 PC에서 접근하려면 터널을 사용하세요.

아래 셀을 실행하면 `public_url`이 출력됩니다.

In [ ]:
# ngrok (선택)
!pip -q install pyngrok
from pyngrok import ngrok

# (선택) ngrok authtoken이 있으면 아래처럼 설정
# ngrok.set_auth_token('YOUR_NGROK_AUTHTOKEN')

public_url = ngrok.connect(8000, 'http')
public_url

## 2) 참고
- MFDS/DUR API 키는 `backend/.env`를 업로드하지 말고, Colab Secret/환경변수로 주입하는 걸 권장합니다.
- RAG는 `frontend/src/data/*.json`을 소스로 사용합니다(번들에 포함됨).